# Project 1: NYC Yellow Taxi Data Detective
**CS672 — Fall 2026 | Due October 16, 2026**

Name: **Add your name**

This is a starter notebook. Run cells in order and replace the writing prompts with your own findings. No analysis has been run yet.

## Our question
Which features are related to how long a taxi ride takes?

Think of each row as a little information card for one ride. We will inspect the cards, find mistakes, draw charts, and explain what we learned. We do not need to train a model.

- March 2020: training dataset.
- May 2020: evaluation dataset.
- January 2020: optional extra credit.

## 1. Set up your tools
Use a Python environment compatible with TensorFlow Data Validation (TFDV). Check its official installation guide before installing; package compatibility varies by environment:
https://www.tensorflow.org/tfx/data_validation/install

The assignment requires TFDV, Apache Beam, and GraphViz. GraphViz is also a system application; installing a Python wrapper alone does not install it. Install dependencies in a dedicated environment rather than an unrelated project.


In [ ]:
# Run this in your chosen notebook environment.
%pip install pandas numpy matplotlib seaborn pyarrow tensorflow-data-validation "apache-beam[interactive]" graphviz

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


## 2. Get the taxi information cards
Download **Yellow Taxi** March 2020 and May 2020 Parquet files from the official NYC TLC page and put them in the `data` folder beside this notebook:
https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

Expected filenames: `yellow_tripdata_2020-03.parquet` and `yellow_tripdata_2020-05.parquet`.

Taxi files can be large. Start with one month if memory is limited, and clearly document any sampling you use.


In [ ]:
paths = {"March": DATA_DIR / "yellow_tripdata_2020-03.parquet",
         "May": DATA_DIR / "yellow_tripdata_2020-05.parquet"}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Download these files first: " + ", ".join(missing))
raw = {month: pd.read_parquet(path) for month, path in paths.items()}
display(raw["March"].head())


## 3. Look before cleaning
Count rows, identify data types, and find missing values. A numeric code can represent a category: payment type and location IDs are examples. Record both the stored type and its meaning.

In [ ]:
for month, frame in raw.items():
    print(month, "rows:", len(frame), "columns:", len(frame.columns))
    display(pd.DataFrame({"stored_type": frame.dtypes.astype(str),
                          "missing_count": frame.isna().sum(),
                          "missing_percent": frame.isna().mean() * 100}))
    display(frame.describe(include="all").T)
    print("Exact duplicate rows:", frame.duplicated().sum())


**Your notes:** List numeric, categorical, and datetime features. Explain missing values and whether apparent duplicates really should be removed.

## 4. Calculate ride duration and inspect unusual rides
Duration is drop-off time minus pickup time, in minutes. Inspect extreme values before deciding what to remove. Negative or zero duration cannot represent a positive-duration ride. Very long rides might be real or might be errors.


In [ ]:
def add_duration(frame):
    result = frame.copy()
    for col in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
        result[col] = pd.to_datetime(result[col], errors="coerce")
    result["trip_duration_minutes"] = (
        result["tpep_dropoff_datetime"] - result["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    return result

prepared = {month: add_duration(frame) for month, frame in raw.items()}
for month, frame in prepared.items():
    print(month)
    display(frame[["trip_duration_minutes", "trip_distance"]].describe(
        percentiles=[.01, .5, .95, .99]))
    display(frame.nlargest(5, "trip_duration_minutes"))


## 5. Clean using explicit rules
The rules below are a starting point: require valid timestamps, a positive duration, a nonnegative known distance, and pickup within the intended month. Zero-distance trips are retained for investigation. Duplicate removal and upper outlier cutoffs need justification before adding them.

Apply the same rules to both months. Keep a count of removed rows. Do not silently drop every row with any missing value.


In [ ]:
def clean_month(frame, month_number):
    result = frame.copy()
    rules = {
        "valid timestamps": result["tpep_pickup_datetime"].notna() & result["tpep_dropoff_datetime"].notna(),
        "positive finite duration": np.isfinite(result["trip_duration_minutes"]) & result["trip_duration_minutes"].gt(0),
        "known nonnegative finite distance": np.isfinite(result["trip_distance"]) & result["trip_distance"].ge(0),
        "pickup in intended month": result["tpep_pickup_datetime"].dt.year.eq(2020) & result["tpep_pickup_datetime"].dt.month.eq(month_number),
    }
    keep = pd.Series(True, index=result.index)
    audit = []
    for name, valid in rules.items():
        before = int(keep.sum())
        keep &= valid
        audit.append({"rule": name, "removed_at_this_step": before - int(keep.sum()), "remaining": int(keep.sum())})
    result = result.loc[keep].copy()
    result["pickup_hour"] = result["tpep_pickup_datetime"].dt.hour
    result["pickup_day_of_week"] = result["tpep_pickup_datetime"].dt.dayofweek
    return result, pd.DataFrame(audit)

clean = {}
for month, number in [("March", 3), ("May", 5)]:
    clean[month], audit = clean_month(prepared[month], number)
    print(month)
    display(audit)


## 6. Explore with pictures
For each picture, write what you see and what it might mean. Correlation is an association, not proof that one feature causes another.

In [ ]:
for month, frame in clean.items():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    # Limit the displayed range only; this does not remove rows from the data.
    upper = frame["trip_duration_minutes"].quantile(.99)
    sns.histplot(frame.loc[frame["trip_duration_minutes"].le(upper), "trip_duration_minutes"], bins=50, ax=axes[0])
    axes[0].set_title(f"{month}: duration (through 99th percentile)")
    sample = frame.sample(n=min(5000, len(frame)), random_state=42)
    sns.scatterplot(data=sample, x="trip_distance", y="trip_duration_minutes", alpha=.2, ax=axes[1])
    axes[1].set_title("Distance and duration (sample)")
    frame.groupby("pickup_hour")["trip_duration_minutes"].median().plot(ax=axes[2])
    axes[2].set_ylabel("Median duration (minutes)")
    axes[2].set_title("Duration by pickup hour")
    plt.tight_layout()
    plt.show()

columns = ["trip_distance", "passenger_count", "pickup_hour", "pickup_day_of_week", "trip_duration_minutes"]
display(clean["March"][columns].corr(method="spearman"))


**Your findings:** Which features appear related to duration? Hour and day are cyclic, so a single correlation coefficient can miss patterns. Examine grouped charts too. Do not call correlation a trained model's feature importance.

## 7. Compare March and May
COVID affected this period. Describe observed differences without claiming this comparison alone proves their cause.


In [ ]:
comparison = pd.DataFrame({month: {
    "cleaned_trip_count": len(frame),
    "median_duration_minutes": frame["trip_duration_minutes"].median(),
    "median_distance": frame["trip_distance"].median(),
    "missing_passenger_count_percent": frame["passenger_count"].isna().mean() * 100,
} for month, frame in clean.items()}).T
display(comparison)
comparison[["median_duration_minutes", "median_distance"]].plot.bar(subplots=True, figsize=(9, 6), legend=False)
plt.tight_layout()
plt.show()


## 8. Use TensorFlow Data Validation (required)
Generate graphical statistics for both months. Infer the schema from March and check May against it. Review detected anomalies; an anomaly is something to investigate, not automatic permission to delete data.

These cells require a working TFDV installation. They have not been executed in this starter notebook.


In [ ]:
import tensorflow_data_validation as tfdv

# Use a focused set of columns to start. Expand for your final analysis.
validation_columns = ["trip_distance", "passenger_count", "pickup_hour", "pickup_day_of_week", "trip_duration_minutes"]
train_stats = tfdv.generate_statistics_from_dataframe(clean["March"][validation_columns])
eval_stats = tfdv.generate_statistics_from_dataframe(clean["May"][validation_columns])
tfdv.visualize_statistics(train_stats, eval_stats, lhs_name="March", rhs_name="May")
schema = tfdv.infer_schema(train_stats)
tfdv.display_schema(schema)
anomalies = tfdv.validate_statistics(eval_stats, schema)
tfdv.display_anomalies(anomalies)


**Your TFDV findings:** Explain the statistics and anomalies you observe. Compare them with your pandas findings.

## 9. Prepare numeric inputs
Choose inputs that would actually be available when making a prediction. Drop-off time and final fare are known after the ride and can leak the answer. Distance below represents recorded trip distance; using it for a pickup-time prediction would require an estimated distance instead.

The example below creates numeric inputs, fills missing passenger counts using March only, and leaves duration as a separate target. Extend it with properly encoded categories if needed. Review outliers and justify your final choices first.


In [ ]:
feature_columns = ["trip_distance", "passenger_count", "pickup_hour", "pickup_day_of_week"]
fill_values = clean["March"][feature_columns].median()
X_train = clean["March"][feature_columns].fillna(fill_values).astype(float)
X_eval = clean["May"][feature_columns].fillna(fill_values).astype(float)
y_train = clean["March"]["trip_duration_minutes"].copy()
y_eval = clean["May"]["trip_duration_minutes"].copy()
assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_eval.to_numpy()).all()
print("Training inputs:", X_train.shape, "Evaluation inputs:", X_eval.shape)
display(X_train.head())


## 10. Tell your data story
Replace these prompts after running your analysis:

1. What data problems did you find, and how many rows did each cleaning rule remove?
2. What are your three strongest findings? Point to specific numbers and charts.
3. Which features seem most useful for understanding duration? What are the limitations?
4. How do March and May differ? Why might a future prediction system need new data?
5. What specific next steps do you recommend?

## Optional extra credit
Download January 2020 Yellow Taxi data, apply the same inspection and cleaning approach, and compare January with March as a pre-pandemic baseline. Consider daily counts because months have different lengths.

## Before submitting
- Run every cell from top to bottom and resolve errors.
- Keep charts, tables, and TFDV output visible in this single notebook.
- Replace all writing prompts with your own evidence-based findings.
- Document sampling, thresholds, dependencies, and limitations.
- Do not upload the large taxi data files to GitHub.
